# Tracking walkthrough — from an uploaded clip to per-person positions

This notebook runs the **exact same path** the Flask app uses, one step at a time, and
prints the intermediate data at every stage so you can see where it comes from and
where it can go wrong.

```
 your .mp4
    │  ensure_mp4()            re-encode to H.264 baseline  (step 1)
    ▼
 POST /api/v1/videos           ask the agent for an upload URL      (step 3)
 POST <upload_url>             push bytes into VST  -> sensorId     (step 4)
 POST /api/v1/videos/{id}/complete   <-- THIS runs RTVI-CV          (step 5)
    │
    ├──> POST /chat            video_understanding -> winner number (step 6)
    │
    └──> RTVI-CV ──Kafka──> Elasticsearch  mdx-raw-*                (steps 7-10)
                                  │
                                  ├─ flatten detections             (step 11)
                                  ├─ drop coasted boxes             (step 11)
                                  ├─ merge duplicate tracker ids    (step 13)
                                  └─ one track per person           (step 14)
                                           │
                                           └─ ffmpeg drawbox overlay (step 15)
```

**Two things are worth knowing before you start:**

1. The tracking data does **not** come from the VSS agent. It is read straight out of
   Elasticsearch. Step 8 explains why.
2. Almost every wrong result we hit came from one of four causes, and each has its own
   diagnostic cell here: a stale sensor, reading too early, a dropped `confidence`
   field, or the tracker issuing several ids for one person. Step 17 summarises them.

**How to run it:** top to bottom. Set `RUN_UPLOAD = False` in the config cell to skip
steps 3–6 and work against a clip that is already in Elasticsearch — useful when you
are only debugging the tracking side and don't want to re-ingest anything.

## Step 0 — Configuration and a look at the stack

In [1]:
!pip install requests overlay

In [2]:
import json, os, re, subprocess, sys, time
from collections import Counter, defaultdict
from pathlib import Path

import requests

# --- where this notebook lives -------------------------------------------
REPO = Path.cwd()                       # run the notebook from visual-inspector/
APP  = REPO / "app"
sys.path.insert(0, str(APP))            # so we can import the real app modules

# --- the clip to analyse --------------------------------------------------
VIDEO = Path("~/Documents/video/dance.mp4").expanduser()

# Set False to skip upload/complete/chat and analyse a clip already ingested.
RUN_UPLOAD = True
# VST keys a clip by its *filename*, and the asset RTVI-Embed creates is keyed by
# the stream id VST hands back for that file. Re-running this notebook therefore
# collides with the previous run unless step 2 cleared it - see step 5.
#   True  -> upload as dance_vst_<stamp>.mp4, which cannot collide with anything.
#   False -> keep the readable name and rely on the purge in step 2.
UNIQUE_UPLOAD_NAME = False
# Only used when RUN_UPLOAD is False - the sensorId to read from Elasticsearch.
EXISTING_VIDEO_NAME = "dance_vst"

# --- service endpoints ----------------------------------------------------
AGENT = os.environ.get("VSS_AGENT_URL", "http://127.0.0.1:8000").rstrip("/")
VST   = os.environ.get("VSS_VST_URL",   "http://127.0.0.1:30000").rstrip("/")
ES    = os.environ.get("VSS_ES_URL",    "http://127.0.0.1:9200").rstrip("/")
MDX_INDEX = "mdx-raw-*"

# Every frame timestamp is this base plus the offset into the video.
UPLOAD_TS = os.environ.get("VSS_UPLOAD_TS", "2025-01-01T00:00:00")

print("repo :", REPO)
print("video:", VIDEO, "->", "exists" if VIDEO.is_file() else "MISSING")
print("agent:", AGENT, "\nvst  :", VST, "\nes   :", ES)


repo : /home/nvidia/Documents/mickael_folder/visual-inspector
video: /home/nvidia/Documents/video/dance.mp4 -> exists
agent: http://127.0.0.1:8000 
vst  : http://127.0.0.1:30000 
es   : http://127.0.0.1:9200


Now check that the three services we depend on are actually up. If one of these is
down, the failure later on will look like "no tracking data" rather than "the service
is down", which is exactly the confusion that cost us the most time.

In [3]:
def ping(label, url, ok=lambda r: r.ok):
    try:
        r = requests.get(url, timeout=10)
        state = "UP  " if ok(r) else f"HTTP {r.status_code}"
    except Exception as exc:
        state = f"DOWN ({type(exc).__name__})"
    print(f"  {label:<22} {state:<22} {url}")

print("services")
ping("VSS agent",      f"{AGENT}/openapi.json")
ping("VST",            f"{VST}/vst/api/v1/sensor/list")
ping("Elasticsearch",  f"{ES}/")
ping("Kibana (optional)", "http://127.0.0.1:5601/api/status")

# What is listening locally. This port scan is what originally revealed that
# Elasticsearch + Kafka were on the box at all.
print("\nlistening ports (9200=ES, 9092=Kafka, 5601=Kibana, 8017/8018=RT-VLM)")
print(subprocess.run(["bash","-lc","ss -ltn | awk 'NR>1{print $4}' | sort -u | tail -30"],
                     capture_output=True, text=True).stdout)

services
  VSS agent              UP                     http://127.0.0.1:8000/openapi.json
  VST                    UP                     http://127.0.0.1:30000/vst/api/v1/sensor/list
  Elasticsearch          UP                     http://127.0.0.1:9200/
  Kibana (optional)      HTTP 404               http://127.0.0.1:5601/api/status

listening ports (9200=ES, 9092=Kafka, 5601=Kibana, 8017/8018=RT-VLM)
[::]:30559
[::]:30560
[::]:30561
[::]:30562
[::]:30563
[::]:30564
[::]:30888
[::]:31000
[::]:38063
[::]:38111
[::]:38112
[::]:40557
[::]:41059
[::]:44619
[::]:47017
[::]:5003
[::]:5601
[::]:5901
[::]:6379
*:6443
[::]:7777
[::]:8000
[::]:8011
[::]:8018
*:8086
[::]:8888
[::]:9092
[::]:9200
[::]:9400
[::]:9902



## Step 1 — Re-encode the clip, and note the name it will be stored under

VST only reliably accepts H.264 baseline, so every upload goes through `ensure_mp4()`
first. That function appends `_vst` to the filename — and **the filename is what
identifies the clip in Elasticsearch later**, so it matters more than it looks.

In [4]:
from overlay import ensure_mp4, probe_duration, probe_video

if RUN_UPLOAD:
    MP4 = ensure_mp4(VIDEO)                 # dance.mp4 -> dance_vst.mp4
    BASE_STEM = MP4.stem                    # the name every run of this clip shares
else:
    MP4 = VIDEO
    BASE_STEM = EXISTING_VIDEO_NAME

FPS, WIDTH, HEIGHT = probe_video(MP4)
DURATION = probe_duration(MP4)

# The name VST and Elasticsearch will know this clip by. It does NOT have to be
# the local filename - only the `filename` fields of the upload decide it - so a
# stamped name costs nothing and cannot inherit an earlier run's stream id.
if RUN_UPLOAD:
    UPLOAD_NAME = (f"{BASE_STEM}_{time.strftime('%Y%m%d%H%M%S')}.mp4"
                   if UNIQUE_UPLOAD_NAME else MP4.name)
    VIDEO_NAME = Path(UPLOAD_NAME).stem
else:
    UPLOAD_NAME = None
    VIDEO_NAME = EXISTING_VIDEO_NAME

print(f"local file  : {MP4.name}")
print(f"resolution  : {WIDTH}x{HEIGHT} @ {FPS:.2f} fps")
print(f"duration    : {DURATION:.2f} s  (~{int(DURATION*FPS)} frames)")
print(f"\nUPLOAD_NAME : {UPLOAD_NAME}")
print("             ^ the name VST files the stream under")
print(f"VIDEO_NAME  : {VIDEO_NAME}")
print("             ^ this, not the sensor UUID, is what we query Elasticsearch with")
print(f"BASE_STEM   : {BASE_STEM}")
print("             ^ what step 2 purges: matches this run AND every earlier one")


local file  : dance_vst.mp4
resolution  : 2560x1440 @ 29.97 fps
duration    : 11.03 s  (~330 frames)

UPLOAD_NAME : dance_vst.mp4
             ^ the name VST files the stream under
VIDEO_NAME  : dance_vst
             ^ this, not the sensor UUID, is what we query Elasticsearch with
BASE_STEM   : dance_vst
             ^ what step 2 purges: matches this run AND every earlier one


> **Pitfall #1 — two different identifiers.**
> The upload returns a `sensorId` that is a UUID. Elasticsearch stores the CV frames
> under a field *also called* `sensorId`, but its value is the **video name**
> (`dance_vst`), not that UUID. Querying with the UUID returns zero rows and looks
> exactly like "the CV pipeline never ran". We lost a while to this.
>
> Also note: going through the Flask app prefixes the name with a uuid
> (`a1b2…_dance_vst`), while running these helpers directly does not. Same clip, two
> different names in the index.

## Step 2 — Clear stale sensors before uploading

One upload leaves traces in **three** separate registries, and they are cleaned up
independently:

| registry | keyed by | who reads it |
| --- | --- | --- |
| VST sensor list | stream UUID, with the filename as `name` | `/complete`, RTVI-CV |
| VSS asset store | the **same UUID**, as the asset id | RTVI-Embed (step 5) |
| Elasticsearch `mdx-raw-*` | the **video name** | tracking.py (step 11) |

Re-running this notebook under the same filename is what makes them collide. RTVI-CV
refuses a stream whose camera id already exists:

```
STREAM_ADD_FAIL, Duplicate Camera id
```

and RTVI-Embed refuses an asset whose id already exists:

```
{"code":"AssetAlreadyExists","message":"Asset with id <uuid> already exists."}
```

Either one fails `/complete`, **no CV metadata is produced at all**, and the old code
only printed a warning and carried on. The game still worked, so nothing looked broken
— but there was never any tracking data.

The purge below therefore **verifies** that the sensors actually disappeared instead of
trusting the delete's status code. A survivor here is the direct cause of the step 5
failure.


In [5]:
from process_video import delete_video, list_sensors

sensors = list_sensors()          # GET {VST}/vst/api/v1/sensor/list
print(f"{len(sensors)} sensor(s) registered in VST\n")
for s in sensors[:20]:
    mark = "  <-- same name as our clip" if BASE_STEM in (s.get("name") or "") else ""
    print(f"  {s.get('sensorId','?'):<40} {s.get('name','?')}{mark}")
if len(sensors) > 20:
    print(f"  ... {len(sensors) - 20} more")

# Print the matches separately: the list above is truncated, and a leftover from an
# earlier run can easily be hiding past the cut - which is exactly how the collision
# in step 5 goes unnoticed.
mine = [s for s in sensors if BASE_STEM in (s.get("name") or "")]
print(f"\n{len(mine)} sensor(s) named like '{BASE_STEM}':")
for s in mine:
    print(f"  {s.get('sensorId','?'):<40} {s.get('name','?')}")


2 sensor(s) registered in VST

  2ff648cf-9e7d-44a9-a39c-eb846a8c1333     dance_vst  <-- same name as our clip
  366dd64c-f334-478d-9f3a-bef89fce0bc1     99c77147a1b148a68155e9f159967ad9_dance

1 sensor(s) named like 'dance_vst':
  2ff648cf-9e7d-44a9-a39c-eb846a8c1333     dance_vst


In [6]:
# purge_videos() unrolled, with the verification the library version used to skip:
# it counted successful DELETEs, not sensors that were actually gone afterwards.

def purge_verified(stem):
    """DELETE every sensor whose name contains `stem`; return the survivors."""
    victims = [s for s in list_sensors() if stem in (s.get("name") or "")]
    if not victims:
        print(f"nothing registered under '{stem}' - clean slate")
        return []

    for s in victims:
        sid = s.get("sensorId")
        try:
            r = requests.delete(f"{AGENT}/api/v1/videos/{sid}", timeout=(10, 120))
            print(f"  DELETE {sid}  ({s.get('name')})  ->  {r.status_code} {r.text[:160]}")
        except requests.RequestException as exc:
            print(f"  DELETE {sid}  ({s.get('name')})  ->  FAILED {type(exc).__name__}: {exc}")

    return [s for s in list_sensors() if stem in (s.get("name") or "")]


if RUN_UPLOAD:
    LEFTOVER = purge_verified(BASE_STEM)
    if LEFTOVER:
        print("\n" + "!"*70)
        print(f"{len(LEFTOVER)} sensor(s) SURVIVED the purge:")
        for s in LEFTOVER:
            print("   ", s.get("sensorId"), s.get("name"))
        print("VST may hand one of these ids back for the upload below, and step 5")
        print("then fails with AssetAlreadyExists / Duplicate Camera id.")
        print("Step 5 recovers by re-uploading under a new name; to skip the")
        print("collision entirely, set UNIQUE_UPLOAD_NAME = True in step 0.")
        print("!"*70)
    else:
        print(f"\nverified: no sensor named like '{BASE_STEM}' is left")
else:
    LEFTOVER = []
    print("skipped (RUN_UPLOAD = False)")


  DELETE 2ff648cf-9e7d-44a9-a39c-eb846a8c1333  (dance_vst)  ->  200 {"status":"success","message":"Video '2ff648cf-9e7d-44a9-a39c-eb846a8c1333' deleted successfully","video_id":"2ff648cf-9e7d-44a9-a39c-eb846a8c1333"}

verified: no sensor named like 'dance_vst' is left


In [7]:
# What else could be holding this clip? The asset store RTVI-Embed uses is keyed
# by the VST stream id, and the purge above cannot see it: it lists VST sensors,
# and an asset orphaned under an id whose sensor is already gone appears nowhere.
#
# Do NOT try to clean that up by sweeping the agent's DELETE routes. This listing
# is diagnostic only - /api/v1/videos/{id} is in it, and calling it after the
# upload deletes the live stream, killing the CV run that just started. The fix
# for an orphaned asset is a new id, i.e. a name no earlier run has used.

_spec = requests.get(f"{AGENT}/openapi.json", timeout=15).json()
print("agent DELETE routes that take one id:")
for p, ops in sorted(_spec["paths"].items()):
    if "delete" in ops and p.count("{") == 1:
        print("   ", p)


agent DELETE routes that take one id:
    /api/v1/rtsp-streams/delete/{name}
    /api/v1/videos/{video_id}
    /static/{file_path}


## Step 3 — Upload, part 1: ask the agent where to put the bytes

The agent does not receive the video itself. It hands back a VST upload URL.

In [8]:
if RUN_UPLOAD:
    init = requests.post(f"{AGENT}/api/v1/videos",
                         json={"filename": UPLOAD_NAME}, timeout=(15, 60))
    print("POST", f"{AGENT}/api/v1/videos", "->", init.status_code)
    print(json.dumps(init.json(), indent=2)[:800])
    UPLOAD_URL = init.json()["url"]
else:
    UPLOAD_URL = None
    print("skipped")

POST http://127.0.0.1:8000/api/v1/videos -> 200
{
  "url": "http://172.16.0.189:7777/vst/api/v1/storage/file"
}


## Step 4 — Upload, part 2: push the bytes into VST

This is the nvstreamer chunked protocol. We send the whole file as a single chunk.
The `metadata.timestamp` we send here is the **base time for every frame timestamp**
we will read back out of Elasticsearch later — remember it, step 10 depends on it.

In [9]:
import mimetypes, uuid as _uuid

if RUN_UPLOAD:
    mime = mimetypes.guess_type(UPLOAD_NAME)[0] or "video/mp4"
    with MP4.open("rb") as fh:
        sent = requests.post(
            UPLOAD_URL,
            headers={
                "nvstreamer-chunk-number": "1",
                "nvstreamer-total-chunks": "1",
                "nvstreamer-is-last-chunk": "true",
                "nvstreamer-identifier": _uuid.uuid4().hex,
                "nvstreamer-file-name": UPLOAD_NAME,
            },
            files={"mediaFile": (UPLOAD_NAME, fh, mime)},
            data={"filename": UPLOAD_NAME,
                  "metadata": json.dumps({"timestamp": UPLOAD_TS})},
            timeout=(15, 900),
        )
    print("POST <upload_url> ->", sent.status_code)
    UPLOAD_PAYLOAD = sent.json()
    print(json.dumps(UPLOAD_PAYLOAD, indent=2)[:1200])

    SENSOR_ID = UPLOAD_PAYLOAD["sensorId"]
    print(f"\nSENSOR_ID  : {SENSOR_ID}      <- the UUID, used by the agent")
    print(f"VIDEO_NAME : {VIDEO_NAME}      <- used by Elasticsearch")
else:
    SENSOR_ID, UPLOAD_PAYLOAD = None, None
    print("skipped")

POST <upload_url> -> 200
{
  "bytes": 16176754,
  "chunkCount": "1",
  "chunkIdentifier": "3b2f4e91d2964885b2f42b3f29ae98b0",
  "created_at": "2026-9-16T16:30:27.959Z",
  "filePath": "/home/vst/vst_release/streamer_videos/dance_vst.mp4",
  "filename": "dance_vst",
  "id": "f849fbde-b2db-4aa5-9f2e-c7aac21a011b",
  "sensorId": "f191940e-58d5-488a-af40-6d644aed1e17",
  "streamId": "f191940e-58d5-488a-af40-6d644aed1e17"
}

SENSOR_ID  : f191940e-58d5-488a-af40-6d644aed1e17      <- the UUID, used by the agent
VIDEO_NAME : dance_vst      <- used by Elasticsearch


## Step 5 — `/complete`: the step that actually runs the CV pipeline

This is the single most important call in the notebook. `/complete` is what fans the
clip out to **RTVI-CV** (detection + tracking) and RTVI-Embed. Without it you still
get a working `video_understanding` answer — which is why its failure went unnoticed
for so long — but there is **zero** tracking data.

The cell below deliberately does *not* raise on failure; it prints the response body,
because that body is where the real reason lives. Two bodies mean "an earlier run
still owns this stream id":

```
502  Embedding generation failed ... {"code":"AssetAlreadyExists",
     "message":"Asset with id <uuid> already exists."}      <- the VSS asset store
     STREAM_ADD_FAIL, Duplicate Camera id                   <- the VST sensor list
```

**There is nothing to clean up at this point.** The id in that message is also the id
of the upload you just made, so deleting anything under it deletes the live stream —
and `/complete` may already have started RTVI-CV, which then dies a fraction of a
second into the clip. The only repair is a **new id**, which means a **new name**:
the cell re-uploads the same bytes under a stamped `UPLOAD_NAME` and rewrites
`VIDEO_NAME`, so every step below follows the new registration.


In [10]:
import mimetypes
import uuid as _uuid

COLLISION = re.compile(r"AssetAlreadyExists|Duplicate Camera id|already exists", re.I)


def complete(sensor_id, payload):
    r = requests.post(f"{AGENT}/api/v1/videos/{sensor_id}/complete",
                      json=payload, timeout=(15, 900))
    print("POST /complete ->", r.status_code, "|", r.text[:600])
    return r


def upload_bytes(mp4, name):
    """Steps 3 + 4 again, under `name`. The bytes are the same local file."""
    init = requests.post(f"{AGENT}/api/v1/videos",
                         json={"filename": name}, timeout=(15, 60))
    init.raise_for_status()
    mime = mimetypes.guess_type(name)[0] or "video/mp4"
    with mp4.open("rb") as fh:
        sent = requests.post(
            init.json()["url"],
            headers={"nvstreamer-chunk-number": "1",
                     "nvstreamer-total-chunks": "1",
                     "nvstreamer-is-last-chunk": "true",
                     "nvstreamer-identifier": _uuid.uuid4().hex,
                     "nvstreamer-file-name": name},
            files={"mediaFile": (name, fh, mime)},
            data={"filename": name,
                  "metadata": json.dumps({"timestamp": UPLOAD_TS})},
            timeout=(15, 900),
        )
    sent.raise_for_status()
    payload = sent.json()
    payload["filename"] = name
    return payload


if RUN_UPLOAD:
    payload = {**UPLOAD_PAYLOAD, "filename": UPLOAD_NAME}
    done = complete(SENSOR_ID, payload)
    CV_READY = done.ok

    # The id belongs to an earlier run. Nothing may be deleted under it - that is
    # the live stream, and RTVI-CV may already be writing. Take a new name instead.
    if not CV_READY and COLLISION.search(done.text or ""):
        print("\n-> id is owned by an earlier run; re-uploading under a new name")
        delete_video(SENSOR_ID)              # safe: /complete failed, nothing is writing
        UPLOAD_NAME = f"{BASE_STEM}_{time.strftime('%Y%m%d%H%M%S')}.mp4"
        UPLOAD_PAYLOAD = upload_bytes(MP4, UPLOAD_NAME)
        SENSOR_ID = UPLOAD_PAYLOAD["sensorId"]
        VIDEO_NAME = Path(UPLOAD_NAME).stem
        print(f"   UPLOAD_NAME: {UPLOAD_NAME}")
        print(f"   SENSOR_ID  : {SENSOR_ID}")
        print(f"   VIDEO_NAME : {VIDEO_NAME}   <- every cell below now uses this")
        done = complete(SENSOR_ID, UPLOAD_PAYLOAD)
        CV_READY = done.ok

    print("\ncv_ready =", CV_READY)
    if not CV_READY:
        print("\n" + "!"*70)
        print("RTVI-CV did NOT process this clip. Everything below will be empty.")
        print("The response body above is the reason - read it before re-running.")
        print("Do not delete anything under this sensor id to 'fix' it: that is")
        print("what truncates the CV run. Set UNIQUE_UPLOAD_NAME = True in step 0")
        print("and re-run from step 1 instead.")
        print("!"*70)
else:
    CV_READY = None
    print("skipped")


POST /complete -> 200 | {"message":"Video dance_vst.mp4 successfully uploaded to VST","sensor_id":"f191940e-58d5-488a-af40-6d644aed1e17","filename":"dance_vst.mp4","chunks_processed":0}

cv_ready = True


> **Pitfall #2 — a non-fatal failure that silently removes the whole feature.**
> The original code caught this exception, printed `[warn] /complete failed`, and
> returned the `sensor_id` anyway. The app kept working and the CV data never existed.
> `upload_video()` now returns `(sensor_id, video_name, cv_ready)` so the caller can
> tell the difference between "the game failed" and "there is no tracking" — and so it
> can tell the caller which name the clip was actually registered under.
>
> **Pitfall #2b — the purge cleans one registry, `/complete` trips over another.**
> `AssetAlreadyExists` names a *new* UUID, which looks impossible: the upload just
> created it. It is not — VST reuses the stream id it already holds for a file of
> that name, and an asset orphaned under that id by an earlier run comes straight
> back with it. The step 2 purge cannot see that asset: it lists VST *sensors*, and
> the sensor is already gone.
>
> **Pitfall #2c — "cleaning up" the collision is what truncates the clip.**
> The obvious repair — delete everything under the offending id, then call
> `/complete` again — deletes the stream you just uploaded, because the offending id
> *is* your stream id. `/complete` has usually already started RTVI-CV by then, so
> the tracker dies a couple of tenths of a second into the clip and leaves a stub in
> Elasticsearch. A 15 s clip came back with 0.2 s of tracking and a duplicate-ingest
> warning that way. All cleanup belongs *before* the upload; afterwards, the only
> repair is a new name.
>
> The Flask app never hits this: `upload_video()` registers every clip under
> `<name>_<8 hex>.mp4`. Only this notebook, which keeps the readable name
> `dance_vst` when `UNIQUE_UPLOAD_NAME` is off, can collide with itself.


## Step 6 — The chat call (the *other* consumer of this upload)

This is the original MoveMatch path, unrelated to tracking, shown here so you can see
both branches come from one upload. The agent is asked a natural-language question and
answers in prose; the app then scrapes a digit out of it.

In [11]:
from process_video import discover_endpoint, _build_payload, _extract_text

ENDPOINT = discover_endpoint()      # picks /chat, /generate, ... from the OpenAPI spec
print("chat endpoint:", ENDPOINT)

spec = requests.get(f"{AGENT}/openapi.json", timeout=15).json()
print("\nall agent routes:")
for p in sorted(spec["paths"]):
    print("   ", p, list(spec["paths"][p].keys()))

chat endpoint: /chat

all agent routes:
    /api/v1/rtsp-streams/add ['post']
    /api/v1/rtsp-streams/delete/{name} ['delete']
    /api/v1/videos ['post']
    /api/v1/videos-for-search/{filename} ['put']
    /api/v1/videos/{sensor_id}/complete ['post']
    /api/v1/videos/{video_id} ['delete']
    /auth/redirect ['get']
    /chat ['post']
    /chat/stream ['post']
    /evaluate ['post']
    /evaluate/item ['post']
    /evaluate/job/last ['get']
    /evaluate/job/{job_id} ['get']
    /evaluate/jobs ['get']
    /executions/{execution_id} ['get']
    /executions/{execution_id}/interactions/{interaction_id}/response ['post']
    /generate ['post']
    /generate/async ['post']
    /generate/async/job/{job_id} ['get']
    /generate/full ['post']
    /generate/stream ['post']
    /mcp/client/tool/list ['get']
    /mcp/client/tool/list/per_user ['get']
    /static/{file_path} ['post', 'put', 'get', 'delete']
    /v1/chat ['post']
    /v1/chat/completions ['post']
    /v1/chat/stream ['post']
 

In [12]:
PROMPT = (
    "Assign a number to each person from left to right.\n"
    "Who is the first person to raise their hand and then turn around, in that order?\n"
    "Answer with only the person's number."
)

if RUN_UPLOAD:
    composed = (
        f"Analyse the video with sensor_id `{SENSOR_ID}` (uploaded file: {UPLOAD_NAME}).\n\n"
        f"User question:\n{PROMPT}\n\n"
        "Use the video_understanding tool with that exact sensor_id and omit "
        "start/end timestamps to cover the whole video. Answer only the question."
    )
    body = _build_payload(ENDPOINT, composed)
    print("request body:\n", json.dumps(body, indent=2)[:600], "\n")

    chat = requests.post(f"{AGENT}{ENDPOINT}", json=body, timeout=(15, 1800))
    print("->", chat.status_code)
    RAW_ANSWER = chat.json()
    print("\nRAW envelope:\n", json.dumps(RAW_ANSWER, indent=2)[:1500])
else:
    RAW_ANSWER = None
    print("skipped")

request body:
 {
  "messages": [
    {
      "role": "user",
      "content": "Analyse the video with sensor_id `f191940e-58d5-488a-af40-6d644aed1e17` (uploaded file: dance_vst.mp4).\n\nUser question:\nAssign a number to each person from left to right.\nWho is the first person to raise their hand and then turn around, in that order?\nAnswer with only the person's number.\n\nUse the video_understanding tool with that exact sensor_id and omit start/end timestamps to cover the whole video. Answer only the question."
    }
  ]
} 



-> 200

RAW envelope:
 {
  "id": "3405fc42-4804-49bb-8862-9f843a2cd5cf",
  "object": "chat.completion",
  "model": "unknown-model",
  "created": 1789576246,
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "\n\n<agent-think><agent-think-step title=\"1 - Thought\">Plan:   1. `video_understanding(sensor_id=\"f191940e-58d5-488a-af40-6d644aed1e17\", user_prompt=\"Assign a number to each person from left to right. Who is the first person to raise their hand and then turn around, in that order? Answer with only the person's number.\")` \u2014 omit start/end timestamps to analyze the entire uploaded video `dance_vst.mp4`.</agent-think-step>\n<agent-think-step title=\"2 - Thought\">The user wants me to analyze a video with sensor_id `f191940e-58d5-488a-af40-6d644aed1e17` (uploaded file: dance_vst.mp4). They want me to:  1. Assign a number to each person from left to right 2. Find who is the first person to raise their hand and then tur

In [13]:
if RAW_ANSWER is not None:
    text = _extract_text(RAW_ANSWER)
    print("extracted text:\n", text, "\n")

    match = re.search(r"\b(\d+)\b", text or "")
    print("regex-scraped winner:", match.group(1) if match else None)
    print("\nNote how fragile that last line is: the FIRST number anywhere in the prose")
    print("wins. A sentence starting 'In the 5 second clip...' yields winner = 5.")

extracted text:
 

<agent-think><agent-think-step title="1 - Thought">Plan:   1. `video_understanding(sensor_id="f191940e-58d5-488a-af40-6d644aed1e17", user_prompt="Assign a number to each person from left to right. Who is the first person to raise their hand and then turn around, in that order? Answer with only the person's number.")` — omit start/end timestamps to analyze the entire uploaded video `dance_vst.mp4`.</agent-think-step>
<agent-think-step title="2 - Thought">The user wants me to analyze a video with sensor_id `f191940e-58d5-488a-af40-6d644aed1e17` (uploaded file: dance_vst.mp4). They want me to:  1. Assign a number to each person from left to right 2. Find who is the first person to raise their hand and then turn around, in that order 3. Answer with only the person's number  The execution plan says to use `video_understanding` tool with that exact sensor_id and omit start/end timestamps to cover the whole video.  Let me call the video_understanding tool with the specified

> **Pitfall #3 — the VLM answer is prose, and the winner is a regex.**
> This is the part of the app that existed before tracking. It has no per-person
> identity and no positions — which is the whole reason for the rest of this notebook.
> If the winner number looks wrong, print `text` above before blaming the model.

## Step 7 — Wait for RTVI-CV to finish writing

RTVI-CV processes the clip **asynchronously, and slower than real time on this box**.
If you query Elasticsearch as soon as the first rows appear you get a truncated
timeline — this is how an 11-second clip ended up with 0.33 s of tracking.

The loop below is `tracking.wait_for_cv()` unrolled so you can watch the count climb.

In [30]:
from tracking import cv_progress

deadline = time.time() + 300
last, stable = -1, 0
print(f"polling Elasticsearch for '{VIDEO_NAME}' (clip is {DURATION:.1f}s)\n")

while time.time() < deadline:
    count, latest = cv_progress(VIDEO_NAME)
    print(f"  t+{time.time()%1000:6.1f}s   frames={count:<6} latest={latest if latest is None else round(latest,2)}s")

    if latest is not None and latest >= DURATION - 0.5:
        print("\n-> CV covered the whole clip.")
        break
    if count and count == last:
        stable += 1
        if stable >= 3:
            print("\n-> frame count stopped growing; CV is done (or stalled).")
            break
    else:
        stable, last = 0, count
    time.sleep(3)
else:
    print("\n-> timed out; continuing with whatever landed.")

polling Elasticsearch for 'dance_vst' (clip is 11.0s)

  t+ 215.0s   frames=0      latest=Nones
  t+ 218.0s   frames=0      latest=Nones
  t+ 221.0s   frames=0      latest=Nones
  t+ 224.0s   frames=0      latest=Nones
  t+ 227.0s   frames=0      latest=Nones
  t+ 230.0s   frames=0      latest=Nones
  t+ 233.0s   frames=0      latest=Nones
  t+ 236.0s   frames=0      latest=Nones
  t+ 239.0s   frames=0      latest=Nones
  t+ 242.0s   frames=0      latest=Nones
  t+ 245.1s   frames=0      latest=Nones
  t+ 248.1s   frames=0      latest=Nones
  t+ 251.1s   frames=0      latest=Nones
  t+ 254.1s   frames=0      latest=Nones
  t+ 257.1s   frames=0      latest=Nones
  t+ 260.1s   frames=0      latest=Nones
  t+ 263.1s   frames=0      latest=Nones
  t+ 266.1s   frames=0      latest=Nones
  t+ 269.1s   frames=0      latest=Nones
  t+ 272.1s   frames=0      latest=Nones
  t+ 275.1s   frames=0      latest=Nones
  t+ 278.1s   frames=0      latest=Nones
  t+ 281.1s   frames=0      latest=Nones
  

KeyboardInterrupt: 

> **Pitfall #4 — deleting the sensor kills the data.**
> Deleting a sensor also drops its CV frames from Elasticsearch. The old cleanup ran in
> a `finally:` block, so it deleted the sensor *while RTVI-CV was still writing* —
> truncating the stream and wiping most of what had been produced. Sensor deletion is
> now opt-in via `DELETE_SENSORS=1`.

## Step 8 — Where the data actually lives, and why not the agent

VSS's own tool for this is `attribute_search` on the agent. Its result rows
(`AttributeSearchMetadata`) carry exactly the right fields:

```
sensor_id, object_id, object_type, frame_timestamp, bbox, behavior_score
```

But read its schema descriptions — `frame_timestamp` is *"Best frame timestamp"*,
`start_time`/`end_time` are *"earliest/latest from duplicates"*. It is a **similarity
search**: it deduplicates to one best-frame row per object. That is precisely the
timeline we need, aggregated away by design.

Run the cell to see it for yourself: one row per person, not a trajectory.

In [ ]:
attr = requests.post(f"{AGENT}/api/v1/attribute_search",
                     json={"query": "person",
                           "video_sources": [VIDEO_NAME],
                           "top_k": 50},
                     timeout=(15, 300))
print("->", attr.status_code)
rows = attr.json().get("value", []) if attr.ok else []
print(f"{len(rows)} row(s) returned\n")
for r in rows[:6]:
    print(json.dumps({k: r.get(k) for k in
                      ("object_id","object_type","frame_timestamp","start_time","end_time","bbox")},
                     indent=2))
print("\nOne row per object with a 'best' frame -> no per-timestep positions.")
print("This is the right tool for 'find the person in the red shirt', the wrong one")
print("for 'where was everyone, frame by frame'.")

Now the store underneath it. RTVI-CV emits per-frame detections onto Kafka, which land
in Elasticsearch as **MDX** (Metropolis Data eXchange) documents.

In [ ]:
cat = requests.get(f"{ES}/_cat/indices?v&s=docs.count:desc&h=index,docs.count,store.size",
                   timeout=30)
print(cat.text[:2000])
print("\nmdx-raw-* is the per-frame CV stream. It is large (~600MB) because every")
print("object carries an embedding vector - which is why our queries exclude them.")

## Step 9 — What a single raw MDX document looks like

One document = one frame. This is the ground truth everything downstream is derived
from, so it is worth reading carefully.

In [ ]:
one = requests.post(f"{ES}/{MDX_INDEX}/_search",
                    params={"size": 1},
                    json={"query": {"term": {"sensorId.keyword": VIDEO_NAME}},
                          "sort": [{"timestamp": {"order": "asc"}}]},
                    timeout=(15, 120)).json()

hits = one.get("hits", {}).get("hits", [])
if not hits:
    print(f"No documents for '{VIDEO_NAME}'. Check the name aggregation in the next cell.")
else:
    doc = hits[0]["_source"]
    # Strip embeddings so the print is readable.
    slim = {k: v for k, v in doc.items() if "embedding" not in k.lower()}
    for obj in slim.get("objects", []):
        for k in list(obj):
            if "embedding" in k.lower():
                obj[k] = "<... vector elided ...>"
    print(json.dumps(slim, indent=2)[:2500])
    print("\ntop-level fields:", sorted(doc))

Reading that document:

| field | meaning |
| --- | --- |
| `sensorId` | the **video name** (not the UUID) |
| `id` | frame number |
| `timestamp` | `VSS_UPLOAD_TS` + offset into the clip — subtract the base to get video time |
| `objects[].id` | **the tracker id** — grouping by this is what gives one timeline per person |
| `objects[].type` | `Person`, `Pallet`, ... |
| `objects[].bbox` | `leftX / topY / rightX / bottomY`, in **pixels** |
| `objects[].confidence` | detector confidence; **non-positive means the box was predicted, not detected** |

Note `sensorId` is mapped as `text`, so term queries and sorts must use
`sensorId.keyword`. Using the plain field silently returns nothing.

In [ ]:
# Which video names exist in the index, and how many frames each has.
agg = requests.post(f"{ES}/{MDX_INDEX}/_search",
                    params={"size": 0},
                    json={"aggs": {"names": {"terms": {"field": "sensorId.keyword", "size": 50}}}},
                    timeout=(15, 120)).json()

print("frames per video name in the index:\n")
for b in agg["aggregations"]["names"]["buckets"]:
    mark = "   <-- ours" if b["key"] == VIDEO_NAME else ""
    print(f"  {b['doc_count']:>7}  {b['key']}{mark}")

## Step 10 — The double-ingest check (run this before trusting anything)

If a clip was ingested twice under the same name, **every person appears twice at every
timestamp**. That looks identical to "the tracker is issuing duplicate ids", but the
fixes are opposite: merging ids here would quietly halve your real detections.

Compare total documents against distinct timestamps. They should be about equal.

In [ ]:
chk = requests.post(f"{ES}/{MDX_INDEX}/_search",
                    params={"size": 0},
                    json={"query": {"term": {"sensorId.keyword": VIDEO_NAME}},
                          "aggs": {"instants": {"cardinality": {"field": "timestamp"}}}},
                    timeout=(15, 120)).json()

total    = chk["hits"]["total"]["value"]
instants = int(chk["aggregations"]["instants"]["value"])
expected = int((DURATION or 0) * FPS)

print(f"documents          : {total}")
print(f"distinct timestamps: {instants}")
print(f"expected frames    : ~{expected}   ({DURATION:.1f}s x {FPS:.1f}fps)")

ratio = total / instants if instants else 0
print(f"\ndocs / instants    : {ratio:.2f}")
if ratio > 1.5:
    print("\n>> DOUBLE INGEST. This clip is in the index more than once.")
    print("   No IoU threshold fixes this - delete the sensor and upload once.")
elif total < expected * 0.5:
    print("\n>> TRUNCATED. RTVI-CV stopped early, or you read before it finished.")
else:
    print("\n>> Looks healthy.")

## Step 11 — Fetch every frame and flatten it into detections

Now the real query. Note `_source_includes`: without it each response drags the
embedding vectors along and the query is enormous.

In [ ]:
SOURCE_FIELDS = ("timestamp,id,sensorId,"
                 "objects.id,objects.type,objects.bbox,objects.confidence")
#                                          ^^^^^^^^^^^^^^^^^^^^
# This one was MISSING originally - see the pitfall note below.

resp = requests.post(f"{ES}/{MDX_INDEX}/_search",
                     params={"size": 10000, "_source_includes": SOURCE_FIELDS},
                     json={"query": {"term": {"sensorId.keyword": VIDEO_NAME}},
                           "sort": [{"timestamp": {"order": "asc"}}]},
                     timeout=(15, 120)).json()

FRAMES = [h["_source"] for h in resp["hits"]["hits"]]
print(f"fetched {len(FRAMES)} of {resp['hits']['total']['value']} frame documents")
print("\nfirst frame:\n", json.dumps(FRAMES[0], indent=2)[:900])

In [ ]:
from datetime import datetime, timezone

def base_epoch():
    stamp = datetime.fromisoformat(UPLOAD_TS)
    if stamp.tzinfo is None:
        stamp = stamp.replace(tzinfo=timezone.utc)   # explicit UTC, see note below
    return stamp.timestamp()

BASE = base_epoch()

def offset_seconds(ts):
    m = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    return m.timestamp() - BASE

# Flatten to (t, track_id, bbox_pixels, confidence) with NOTHING filtered yet.
ALL_DETECTIONS, types_seen = [], Counter()
for f in FRAMES:
    t = offset_seconds(f["timestamp"])
    for o in f.get("objects") or []:
        types_seen[o.get("type")] += 1
        b = o.get("bbox") or {}
        if not all(k in b for k in ("leftX","topY","rightX","bottomY")):
            continue
        ALL_DETECTIONS.append((
            t, str(o.get("id")),
            (float(b["leftX"]), float(b["topY"]), float(b["rightX"]), float(b["bottomY"])),
            float(o.get("confidence") or 0.0),
        ))

print("object types in the clip:", dict(types_seen))
print(f"\n{len(ALL_DETECTIONS)} raw detections, {len({d[1] for d in ALL_DETECTIONS})} distinct tracker ids")
print(f"time span: {min(d[0] for d in ALL_DETECTIONS):.2f}s .. {max(d[0] for d in ALL_DETECTIONS):.2f}s")
print("\nfirst 5:")
for d in ALL_DETECTIONS[:5]:
    print(f"  t={d[0]:6.2f}s  id={d[1]:>3}  conf={d[3]:+.3f}  bbox={tuple(round(v,1) for v in d[2])}")

> **The timestamp base is parsed as UTC on purpose.** `VSS_UPLOAD_TS` carries no
> timezone while the frame timestamps end in `Z`. Letting Python apply the local zone
> would skew every offset by this machine's UTC offset — on a box set to CEST that is
> a two-hour shift, and every box would land on the wrong frame.

### The confidence filter — and the bug that made it eat everything

A non-positive `confidence` means the detector did not fire on that frame and the
tracker **predicted** the position. Those coasted boxes are the ones that drift off a
person and land on someone else, so they are dropped.

In [ ]:
conf = [d[3] for d in ALL_DETECTIONS]
coasted = [c for c in conf if c <= 0.0]
real    = [c for c in conf if c > 0.0]

print(f"detections          : {len(conf)}")
print(f"  coasted (<= 0)    : {len(coasted)}   <- dropped by default")
print(f"  detected  (> 0)   : {len(real)}")
if real:
    print(f"  confidence range  : {min(real):.3f} .. {max(real):.3f}")

DETECTIONS = [d for d in ALL_DETECTIONS if d[3] > 0.0]
print(f"\nkept {len(DETECTIONS)} detections, {len({d[1] for d in DETECTIONS})} tracker ids")

> **Pitfall #5 — filtering on a field you never fetched.**
> `_SOURCE_FIELDS` originally did **not** list `objects.confidence`. Elasticsearch
> stripped the field before it reached the parser, so `obj.get("confidence")` was
> always `None` → `0.0` → *every* detection failed the `<= MIN_CONFIDENCE` test. The
> error read:
>
> ```
> 657 CV frames found for 'dance_vst' but no usable Person detections
> (3497 dropped as low-confidence). Detected types: ['Person']
> ```
>
> "Detected types: ['Person']" was the tell — the people were right there. If you ever
> see a filter reject 100% of rows, check that the field is in `_source_includes`.

## Step 12 — How many boxes does each person actually wear?

If the tracker were perfect, the number of boxes in a frame would equal the number of
people. It is not: when the tracker loses and re-acquires someone it issues a **fresh
id**, while the stale track coasts along on top.

In [ ]:
per_frame = Counter()
for t, tid, bbox, c in DETECTIONS:
    per_frame[round(t, 3)] += 1

hist = Counter(per_frame.values())
print("boxes per frame:")
for n in sorted(hist):
    print(f"  {n:>2} box(es)  {'#' * min(60, hist[n])}  {hist[n]} frames")

lifetime = Counter(tid for _, tid, _, _ in DETECTIONS)
print(f"\n{len(lifetime)} tracker ids, by how long they live:")
for tid, n in lifetime.most_common(20):
    ts = [d[0] for d in DETECTIONS if d[1] == tid]
    print(f"  id {tid:>4}  {n:>5} detections   t={min(ts):6.2f}..{max(ts):6.2f}s")

## Step 13 — Merging duplicate tracker ids

Each frame is treated as a small non-maximum suppression: the longest-lived,
most-confident box wins, and any box overlapping it by more than `IOU_MERGE` is
recorded as a duplicate *of it*. An id that loses this way for most of its life gets
folded into its winner; an id that only occasionally overlaps (two people genuinely
standing close) is left alone — that is what `MERGE_DOMINANCE` guards.

Merging **ids** rather than just deleting boxes is what keeps one stable label for the
whole clip; deleting the box alone would leave the duplicate id alive and the person's
label flickering between the two.

In [ ]:
import tracking
from tracking import _merge_duplicate_ids, _iou

print(f"IOU_MERGE={tracking.IOU_MERGE}  MERGE_DOMINANCE={tracking.MERGE_DOMINANCE}  "
      f"MIN_CONFIDENCE={tracking.MIN_CONFIDENCE}")

canonical = _merge_duplicate_ids(DETECTIONS)

merged = {k: v for k, v in canonical.items() if k != v}
print(f"\n{len(set(canonical))} track(s) after merging, from {len(canonical)} raw ids")
print(f"{len(merged)} id(s) folded in:")
for dup, winner in sorted(merged.items(), key=lambda kv: int(kv[1]) if kv[1].isdigit() else 0):
    print(f"   id {dup:>4}  ->  id {winner}")
if not merged:
    print("   (none - either the tracker was clean, or IOU_MERGE is too high)")

## Step 14 — The finished tracks

`fetch_tracks()` does everything above in one call and additionally: keeps one box per
person per frame (the most confident), drops flicker tracks shorter than
`MIN_TRACK_POINTS` / `MIN_TRACK_SECONDS`, and normalises the pixel bboxes to 0–1 so the
overlay is resolution-independent.

In [ ]:
from tracking import fetch_tracks

TRACKS = fetch_tracks(VIDEO_NAME, (WIDTH, HEIGHT))
print(f"{len(TRACKS)} track(s)\n")

for i, tr in enumerate(TRACKS):
    pts = tr["points"]
    span = pts[-1]["t"] - pts[0]["t"]
    fx = (pts[0]["bbox"][0] + pts[0]["bbox"][2]) / 2
    lx = (pts[-1]["bbox"][0] + pts[-1]["bbox"][2]) / 2
    print(f"  P{i+1}  (tracker id {tr['track_id']:>3})  {len(pts):>4} points  "
          f"t={pts[0]['t']:5.2f}..{pts[-1]['t']:5.2f}s ({span:5.2f}s)  "
          f"centre x {fx:.2f} -> {lx:.2f}")

print("\nThis is the answer to 'where was each person at each timestep'.")
print(f"\nfirst 8 samples of P1:")
for p in TRACKS[0]["points"][:8]:
    x1, y1, x2, y2 = p["bbox"]
    print(f"   t={p['t']:6.2f}s  centre=({(x1+x2)/2:.3f}, {(y1+y2)/2:.3f})  "
          f"bbox=({x1:.3f}, {y1:.3f}, {x2:.3f}, {y2:.3f})")

In [ ]:
# Sanity check worth doing every time: does the track count match the people you can
# see in the clip? Too many almost always means duplicate ids or a double ingest.
EXPECTED_PEOPLE = 7      # <-- set this to what you actually see

print(f"tracks found : {len(TRACKS)}")
print(f"people in clip: {EXPECTED_PEOPLE}")
if len(TRACKS) > EXPECTED_PEOPLE:
    print("\n>> Too many. Re-check step 10 (double ingest) FIRST, then tune step 16.")
elif len(TRACKS) < EXPECTED_PEOPLE:
    print("\n>> Too few. IOU_MERGE may be too low (merging distinct people), or")
    print("   the confidence filter is too aggressive - try TRACKING_MIN_CONFIDENCE=-99.")
else:
    print("\n>> Match.")

### A coarse visual of the timeline

Who is on screen when, and roughly where, without rendering anything.

In [ ]:
WIDTH_CHARS = 90
t0 = min(p["t"] for tr in TRACKS for p in tr["points"])
t1 = max(p["t"] for tr in TRACKS for p in tr["points"])

print(f"presence over time   ({t0:.1f}s ---> {t1:.1f}s)\n")
for i, tr in enumerate(TRACKS):
    row = [" "] * WIDTH_CHARS
    for p in tr["points"]:
        col = int((p["t"] - t0) / max(t1 - t0, 1e-6) * (WIDTH_CHARS - 1))
        # character encodes horizontal position in frame: left . -> right #
        cx = (p["bbox"][0] + p["bbox"][2]) / 2
        row[col] = ".:-=+*#"[min(6, int(cx * 7))]
    print(f"P{i+1:<2}|{''.join(row)}|")
print("\nlegend: '.' = left of frame ... '#' = right of frame; blank = not tracked")

## Step 15 — Render the overlay

Drawing goes through ffmpeg's `drawbox`/`drawtext` rather than a Python imaging
library: ffmpeg is already a hard dependency for the VST re-encode, so the overlay adds
nothing to install. The cost is that the filter graph grows with the number of samples,
which is why they get decimated and merged first.

In [ ]:
from overlay import _fit_budget, _build_filters, MAX_FILTER_ENTRIES, HOLD_SECONDS

spans = _fit_budget(TRACKS)
total_spans = sum(len(s) for s in spans)
raw_points  = sum(len(t["points"]) for t in TRACKS)

print(f"raw samples      : {raw_points}")
print(f"spans after merge: {total_spans}   (budget {MAX_FILTER_ENTRIES})")
print(f"each box is held : {HOLD_SECONDS}s past its last sample (stops flicker)")

filters = _build_filters(spans, WIDTH, HEIGHT)
print(f"ffmpeg filters   : {len(filters)}")
print("\nfirst two filters:\n ", "\n  ".join(filters[:2]))

In [ ]:
from overlay import render_overlay

OUT = REPO / "tmp" / f"{VIDEO_NAME}_tracked.mp4"
OUT.parent.mkdir(exist_ok=True)

render_overlay(MP4 if RUN_UPLOAD else VIDEO, TRACKS, OUT)
print("written:", OUT, f"({OUT.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Pull a few frames out so you can check the boxes without leaving the notebook.
from IPython.display import Image, display

grabs = REPO / "tmp" / "grabs"
grabs.mkdir(exist_ok=True)
for old in grabs.glob("*.png"):
    old.unlink()

for i, at in enumerate([t0 + (t1 - t0) * f for f in (0.1, 0.4, 0.7, 0.95)]):
    subprocess.run(["ffmpeg", "-y", "-ss", f"{at:.2f}", "-i", str(OUT),
                    "-frames:v", "1", "-vf", "scale=720:-1",
                    str(grabs / f"{i}.png")], capture_output=True)

for png in sorted(grabs.glob("*.png")):
    display(Image(filename=str(png)))
print("Boxes should sit on the people and keep the same colour/label throughout.")

## Step 16 — Tuning the duplicate suppression

If step 14 gave you too many tracks, sweep the knobs here rather than re-rendering each
time. Each trial re-runs `fetch_tracks` with the module constants overridden.

In [ ]:
import importlib
importlib.reload(tracking)
from tracking import fetch_tracks as _ft

def trial(iou, conf, dom=0.5):
    tracking.IOU_MERGE, tracking.MIN_CONFIDENCE, tracking.MERGE_DOMINANCE = iou, conf, dom
    try:
        return len(tracking.fetch_tracks(VIDEO_NAME, (WIDTH, HEIGHT)))
    except Exception:
        return None

print(f"target: {EXPECTED_PEOPLE} people\n")
print(f"{'coasted':<10} {'IoU':>6}  tracks")
for conf, label in [(-99.0, "kept"), (0.0, "dropped")]:
    for iou in (0.0, 0.35, 0.45, 0.55, 0.65, 0.8):
        n = trial(iou, conf)
        mark = "  <== match" if n == EXPECTED_PEOPLE else ""
        print(f"{label:<10} {iou:>6}  {str(n):>6}{mark}")

# restore defaults
importlib.reload(tracking)

## Step 17 — Summary: everything that produced a wrong result

| Symptom | Real cause | Where to check |
| --- | --- | --- |
| No tracking data at all | `/complete` failed with `Duplicate Camera id`; the old code swallowed it as a warning | step 2, step 5 |
| `/complete` 502 `AssetAlreadyExists` on a UUID issued seconds ago | a sensor the purge failed to remove; VST reuses its stream id and the VSS asset comes back with it | step 2, step 5 |
| Zero rows in Elasticsearch | queried with the sensor **UUID**, or with `sensorId` instead of `sensorId.keyword` | step 1, step 9 |
| 0.33 s of tracking on an 11 s clip | read before RTVI-CV finished, and/or the sensor was deleted in a `finally:` block while it was still writing | step 7 |
| "No usable Person detections" while types show `['Person']` | `objects.confidence` was not in `_source_includes`, so the confidence filter rejected 100% of rows | step 11 |
| One person wearing 2–3 boxes | the tracker re-issues an id after losing someone; the stale track coasts along | steps 12–13 |
| Too many people, immune to IoU tuning | the clip was ingested twice — every person duplicated at every timestamp | **step 10** |
| Every box on the wrong frame | `VSS_UPLOAD_TS` parsed in local time instead of UTC | step 11 |
| Winner number nonsensical | it is a regex over VLM prose; the first digit anywhere wins | step 6 |
| Boxes misaligned on the video | overlay normalised against a different resolution than the ingested copy | step 1 |

**The one that matters most:** run **step 10** before tuning anything. A double ingest
and a re-acquiring tracker produce an identical symptom and need opposite fixes —
merging ids to "fix" a double ingest quietly halves your real detections.

**And the methodological one:** early on, the `Duplicate Camera id` error was read as
proof that the CV pipeline was reachable *through the agent*. It only proved a tracker
had **run**. Those are different claims, and conflating them sent us through several
rounds of probing `attribute_search` before the port scan in step 0 showed where the
data really was.